In [ ]:
# Databricks notebook source


In [ ]:
#Requirements Installation

# MAGIC %pip install openpyxl


In [ ]:
#Imports

from pyspark.sql import SparkSession
from datetime import datetime
import importlib.util
import sys
import os
import yaml
import json
import logging

spark = SparkSession.builder.getOrCreate()
logger = logging.getLogger(__name__)


In [ ]:
#Path Definitions

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
notebook_path = ctx.notebookPath().get()

NOTEBOOK_DIR = f"/Workspace{notebook_path.rsplit('/', 1)[0]}"
WORKSPACE_ROOT = NOTEBOOK_DIR.rsplit('/databricks_bundle/src', 1)[0]
FILES_ROOT = WORKSPACE_ROOT if WORKSPACE_ROOT.endswith('/files') else f"{WORKSPACE_ROOT}/files"
SCRIPTS_DIR  = os.path.abspath(f"{FILES_ROOT}/databricks_bundle/scripts")
INGESTION_DIR = os.path.abspath(f"{FILES_ROOT}/databricks_bundle/drugdev/ingestion_framework")
NOTIFICATION_DIR = os.path.abspath(f"{FILES_ROOT}/databricks_bundle/drugdev/notification_faramework")

print("Notebook Directory:", NOTEBOOK_DIR)
print("Files Root:", FILES_ROOT)
print("Scripts Directory :", SCRIPTS_DIR)
print("Ingestion Directory:", INGESTION_DIR)
print("Notification Directory:", NOTIFICATION_DIR)


In [ ]:
# Workflow Parameters (widgets)

def _widget(name: str, default: str) -> str:
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name).strip()

RAW_BUCKET       = _widget("raw_bucket", "exelixis-clearlake-daplex-dev-us-west-2-441447966705-raw")
STG_BUCKET       = _widget("stg_bucket", "")
CATALOG          = _widget("catalog", "dev-drugdev_da-koios-catalog")
BRONZE_SCHEMA    = _widget("bronze_schema", "dev_drugdev_bronze")
SILVER_SCHEMA    = _widget("silver_schema", "dev_drugdev_silver")
GOLD_SCHEMA      = _widget("gold_schema", "dev_drugdev_gold")
METADATA_CATALOG = _widget("metadata_catalog", "dev-drugdev_da-koios-catalog")
REGISTRY_SCHEMA  = _widget("registry_schema", "dev_drugdev_common")
METADATA_SCHEMA  = _widget("metadata_schema", "dev_drugdev_common")
RUN_DATE         = _widget("run_date", datetime.now().strftime("%Y%m%d"))
ENVIRONMENT      = _widget("environment", "dev")
SOURCE_BUCKET    = _widget("source_bucket", "exelixis-clearlake-daplex-dev-us-west-2-441447966705-raw")
SIMULATION_TYPE  = _widget("simulation_type", "")

run_date        = RUN_DATE
simulation_type = SIMULATION_TYPE
environment     = ENVIRONMENT
source_bucket   = SOURCE_BUCKET

# Propagate to Spark config
spark.conf.set("drugdev.environment",      ENVIRONMENT)
spark.conf.set("drugdev.raw_bucket",       RAW_BUCKET)
spark.conf.set("drugdev.stg_bucket",       STG_BUCKET)
spark.conf.set("drugdev.CATALOG",          CATALOG)
spark.conf.set("drugdev.BRONZE_SCHEMA",    BRONZE_SCHEMA)
spark.conf.set("drugdev.SILVER_SCHEMA",    SILVER_SCHEMA)
spark.conf.set("drugdev.GOLD_SCHEMA",      GOLD_SCHEMA)
spark.conf.set("drugdev.simulation_type",  SIMULATION_TYPE)
spark.conf.set("drugdev.METADATA_CATALOG", METADATA_CATALOG)
spark.conf.set("drugdev.METADATA_SCHEMA",  METADATA_SCHEMA)
spark.conf.set("drugdev.REGISTRY_SCHEMA",  REGISTRY_SCHEMA)
spark.conf.set("drugdev.source_bucket",    SOURCE_BUCKET)
spark.conf.set("drugdev.YML_CONFIG_PATH",  f"{INGESTION_DIR}/configs/drugdev_config.yaml")
spark.conf.set("drugdev.SCHEMA_REGISTRY_PATH", f"{INGESTION_DIR}/ingestion_engine/schema_registry.py")
spark.conf.set("drugdev.RUN_DATE",         run_date)

print("Simulation Type =", simulation_type)
print("RUN_DATE        =", run_date)


In [ ]:
#Import Helper Function

def import_from_path(name, path):
    module_dir = os.path.dirname(os.path.abspath(path))
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)

    # Inject dbutils into the module
    mod.dbutils = dbutils

    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


In [ ]:
# Alerting setup (workflow-only)

def _as_bool(v: str) -> bool:
    return str(v).strip().lower() in {"1", "true", "yes", "y", "on"}

def _get_conf_optional(key: str, default: str = "") -> str:
    return spark.conf.get(key, default).strip()

def _json_list_from_conf(key: str, default: str = "[]") -> list:
    raw = _get_conf_optional(key, default)
    try:
        parsed = json.loads(raw) if raw else []
    except Exception as exc:
        raise ValueError(f"Spark conf {key} must be valid JSON list") from exc
    if not isinstance(parsed, list):
        raise ValueError(f"Spark conf {key} must be a JSON list")
    return parsed

def _json_dict_from_conf(key: str, default: str = "{}") -> dict:
    raw = _get_conf_optional(key, default)
    try:
        parsed = json.loads(raw) if raw else {}
    except Exception as exc:
        raise ValueError(f"Spark conf {key} must be valid JSON object") from exc
    if not isinstance(parsed, dict):
        raise ValueError(f"Spark conf {key} must be a JSON object")
    return parsed

def _build_alerting_config_from_spark() -> dict:
    email_enabled = _as_bool(_get_conf_optional("drugdev.NOTIFY_EMAIL_ENABLED", "false"))
    teams_enabled = _as_bool(_get_conf_optional("drugdev.NOTIFY_TEAMS_ENABLED", "false"))

    missing = []
    if email_enabled:
        for k in [
            "drugdev.NOTIFY_EMAIL_FROM",
            "drugdev.NOTIFY_EMAIL_REPLY_TO",
            "drugdev.NOTIFY_EMAIL_RECIPIENTS_CRITICAL_JSON",
            "drugdev.NOTIFY_EMAIL_RECIPIENTS_HIGH_JSON",
            "drugdev.NOTIFY_EMAIL_RECIPIENTS_MEDIUM_JSON",
        ]:
            if not spark.conf.get(k, "").strip():
                missing.append(k)
    if teams_enabled and not spark.conf.get("drugdev.NOTIFY_TEAMS_WEBHOOK_URL", "").strip():
        missing.append("drugdev.NOTIFY_TEAMS_WEBHOOK_URL")
    if missing:
        raise ValueError("Missing required Spark conf(s) for alerting: " + ", ".join(missing))

    email_cfg = {
        "enabled": email_enabled,
        "smtp_driver": _get_conf_optional("drugdev.NOTIFY_EMAIL_DRIVER", "ses"),
        "from_address": _get_conf_optional("drugdev.NOTIFY_EMAIL_FROM"),
        "reply_to": _get_conf_optional("drugdev.NOTIFY_EMAIL_REPLY_TO"),
        "ses_region": _get_conf_optional("drugdev.NOTIFY_EMAIL_SES_REGION", _get_conf_optional("AWS_DEFAULT_REGION", "us-west-2")),
        "recipients": {
            "critical": _json_list_from_conf("drugdev.NOTIFY_EMAIL_RECIPIENTS_CRITICAL_JSON") if email_enabled else [],
            "high": _json_list_from_conf("drugdev.NOTIFY_EMAIL_RECIPIENTS_HIGH_JSON") if email_enabled else [],
            "medium": _json_list_from_conf("drugdev.NOTIFY_EMAIL_RECIPIENTS_MEDIUM_JSON") if email_enabled else [],
            "low": _json_list_from_conf("drugdev.NOTIFY_EMAIL_RECIPIENTS_LOW_JSON"),
        },
    }

    teams_cfg = {
        "enabled": teams_enabled,
        "webhook_url": _get_conf_optional("drugdev.NOTIFY_TEAMS_WEBHOOK_URL"),
    }

    routing_cfg = _json_dict_from_conf(
        "drugdev.NOTIFY_ROUTING_JSON",
        '{"CRITICAL":{"teams":true},"HIGH":{"teams":true},"MEDIUM":{"teams":false},"LOW":{"teams":false}}',
    )
    domain_recipients_cfg = _json_dict_from_conf("drugdev.NOTIFY_DOMAIN_RECIPIENTS_JSON", "{}")

    return {
        "notifications": {
            "email": email_cfg,
            "teams": teams_cfg,
            "routing": routing_cfg,
            "domain_recipients": domain_recipients_cfg,
        }
    }

if not spark.conf.get("drugdev.CONFIG_PATH", "").strip():
    default_config_path = os.path.abspath(f"{NOTEBOOK_DIR}/../../config/environments/{environment}.yaml")
    spark.conf.set("drugdev.CONFIG_PATH", default_config_path)

alert_notifier = import_from_path(
    "alert_notifier",
    f"{NOTIFICATION_DIR}/alert_notifier.py"
)
alert_cfg = _build_alerting_config_from_spark()
notifier = alert_notifier.AlertNotifier(alert_cfg)


In [ ]:
# Load drugdev config and schema registry

config_path = f"{INGESTION_DIR}/configs/drugdev_config.yaml"
with open(config_path, "r") as f:
    drugdev_config = yaml.safe_load(f)

schema_reg = import_from_path(
    "schema_registry",
    f"{INGESTION_DIR}/ingestion_engine/schema_registry.py"
)


In [ ]:
# Create metadata tables and volumes (idempotent)

METADATA_CATALOG = spark.conf.get("drugdev.METADATA_CATALOG")
REGISTRY_SCHEMA  = spark.conf.get("drugdev.REGISTRY_SCHEMA")
METADATA_SCHEMA  = spark.conf.get("drugdev.METADATA_SCHEMA")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{REGISTRY_SCHEMA}.dataset_registry (
  dataset_id STRING NOT NULL,
  domain_name STRING,
  data_product_name STRING,
  dataset_name STRING,
  dataset_version STRING,
  frequency STRING,
  owner_team STRING,
  owner_email STRING,
  criticality STRING,
  contains_pii BOOLEAN,
  data_classification STRING,
  lifecycle_status STRING,
  retention_days INT,
  created_at TIMESTAMP,
  is_active BOOLEAN
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.ingestion_config (
  config_id STRING NOT NULL,
  dataset_id STRING NOT NULL,
  environment STRING,
  source_bucket STRING,
  source_path STRING,
  file_format STRING,
  delimiter STRING,
  file_encoding STRING,
  load_type STRING,
  file_name STRING,
  row_tag STRING,
  ingestion_mode STRING,
  primary_keys STRING,
  watermark_column STRING,
  checkpoint_location STRING,
  schema_location STRING,
  schema_strategy STRING,
  optimize_write BOOLEAN,
  auto_compact BOOLEAN,
  is_active BOOLEAN
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{REGISTRY_SCHEMA}.dataset_tags (
  tag_id STRING NOT NULL,
  dataset_id STRING NOT NULL,
  tag_key STRING,
  tag_value STRING,
  created_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{REGISTRY_SCHEMA}.dataset_dependencies (
  dependency_id STRING NOT NULL,
  dataset_id STRING NOT NULL,
  depends_on_dataset_id STRING,
  dependency_type STRING,
  created_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.ingestion_runtime_state (
  runtime_id STRING NOT NULL,
  dataset_id STRING NOT NULL,
  environment STRING,
  last_run_status STRING,
  records_ingested BIGINT,
  files_processed INT,
  last_run_start_time TIMESTAMP,
  last_run_end_time TIMESTAMP,
  failure_reason STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.dq_sla_config (
  dq_id STRING NOT NULL,
  dataset_id STRING NOT NULL,
  dq_enabled BOOLEAN,
  rule_set_name STRING,
  row_count_min BIGINT,
  row_count_max BIGINT,
  freshness_minutes INT,
  null_threshold_pct DOUBLE,
  duplicate_threshold_pct DOUBLE,
  anomaly_detection_enabled BOOLEAN,
  fail_action STRING,
  alert_channel STRING,
  escalation_contact STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.schema_registry (
  dataset_id STRING NOT NULL,
  schema_json STRING NOT NULL,
  version INT,
  is_active BOOLEAN,
  created_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"CREATE VOLUME IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.schemas")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.checkpoints")


In [ ]:
# Load ingestion modules

metadata_loader = import_from_path(
    "metadata_loader",
    f"{INGESTION_DIR}/metadata_service/metadata_loader.py"
)

autoloader_engine = import_from_path(
    "autoloader_engine",
    f"{INGESTION_DIR}/ingestion_engine/autoloader_engine.py"
)


In [ ]:
# Execute Bronze extraction workflow with alerting

try:
    metadata_loader.main()

    RAW_BUCKET = spark.conf.get("drugdev.raw_bucket")
    autoloader_engine.main(simulation_type)

    notifier.send_alert(
        alert_type="BRONZE_COMPLETION",
        severity="LOW",
        title=f"Bronze Completion [{environment}] - SUCCESS",
        message=f"Bronze ingestion completed for run_date={run_date} simulation_type={simulation_type or 'NA'}.",
        context={
            "run_date": run_date,
            "simulation_type": simulation_type or "NA",
            "raw_bucket": RAW_BUCKET,
        },
    )
except Exception as exc:
    logger.error("Bronze workflow failed: %s", exc, exc_info=True)
    try:
        notifier.send_alert(
            alert_type="BRONZE_COMPLETION",
            severity="CRITICAL",
            title=f"Bronze Completion [{environment}] - FAILED",
            message=f"Bronze ingestion failed: {exc}",
            context={
                "run_date": run_date,
                "simulation_type": simulation_type or "NA",
            },
        )
    except Exception as alert_exc:
        logger.warning("Failed to send bronze completion alert: %s", alert_exc)
    raise
